# 01 — Data Audit

Before any visualization or modeling, this notebook asks one question: can we trust this data? We look at structure, missing values, class balance, and potential leakage risks before touching anything else.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

Matplotlib is building the font cache; this may take a moment.


In [3]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

print("Libraries loaded.")

Libraries loaded.


In [8]:
# column names are from data source, check data_card.md
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

In [9]:
# upload data into pandas dataframes
train = pd.read_csv(
    "../data/raw/adult.data",
    names=columns,
    skipinitialspace=True
)

test = pd.read_csv(
    "../data/raw/adult.test",
    names=columns,
    skipinitialspace=True,
    skiprows=1
)

In [10]:
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

Train shape: (32561, 15)
Test shape:  (16281, 15)


## 2. Basic Structure

In [14]:
train.tail()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K
32560,52,Self-emp-inc,287927,HS-grad,9,Married-civ-spouse,Exec-managerial,Wife,White,Female,15024,0,40,United-States,>50K


In [15]:
train.dtypes

age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
income              str
dtype: object

In [16]:
train.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


### Observations 1 - Basic Structure

- **age:** ranges from 17 to 90, no obvious outliers. Mean app. 38
- **fnlwgt:** census sampling weight with very high variance, 12K to 1.48M.
    This column will be dropped before modeling. It reflects how the census was collected, not individual characteristics.
- **education-num:** ranges 1 - 16, mean ~10. Numerical encoding of education level - redundant with the education column. It should be examined, and one will be dropped.
- **capital-gain:** mean ~1077 but 75th percentile is 0. Max is 99,999, likely a cap value, not a real observation. Heavily zero-inflated. Need log transformation or special treatment.
- **capital-loss:** same pattern, 75th percentile is 0, heavily zero-inflated.
- **hours-per-week:** ranges 1 to 99. Mean ~40 as expected. Max of 99 is suspicious. It could be a cap or data entry artifact.

In [25]:
train.income.value_counts(normalize=True)

income
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64

### Observations 2 - Class Balance

~75% of instances are <=50K, ~25% are >50K. This imbalance is meaningful a naive model predicting <=50K for everyone would achieve 75% accuracy. Accuracy alone is therefore a misleading metric for this dataset. Precision-recall curves and F1 score will be more informative than ROC-AUC alone.

_____________________________________

## 3. Missing Values

Missing values in this dataset are encoded as `?` — not standard nulls.
A naive `df.isnull().sum()` would return zero and give false confidence.

In [26]:
# standard null check -- will show nothing
print("Standard null check:")
print(train.isnull().sum())

Standard null check:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


In [27]:
# actual missing values encoded as "?"
print("Actual missing values (encoded as '?'):")
missing = (train == "?").sum()
print(missing[missing > 0])
print()
print("As percentage of total rows:")
print((missing[missing > 0] / len(train) * 100).round(2))

Actual missing values (encoded as '?'):
workclass         1836
occupation        1843
native-country     583
dtype: int64

As percentage of total rows:
workclass         5.64
occupation        5.66
native-country    1.79
dtype: float64


### Observations 3 — Missing Values

Standard `isnull()` returns zero — missing values are encoded as `?` and require explicit detection.

Three columns contain missing values:
- **workclass:** 1,836 missing (5.64%) 
- **occupation:** 1,843 missing (5.66%) 
- **native-country:** 583 missing (1.79%)

The near-identical counts in `workclass` and `occupation` are not coincidental — they are likely the same rows. Someone without a recorded 
workclass probably also has no recorded occupation. This needs verification. As you see, this is exploration step, we cannot be sure unless we examined in detail.

In [31]:
# are workclass and occupation missing in the same rows?
both_missing = ((train["workclass"] == "?") & (train["occupation"] == "?")).sum()
print(f"Rows where both workclass AND occupation are missing: {both_missing}")

Rows where both workclass AND occupation are missing: 1836


The 7 rows where `occupation` is missing but `workclass` is not are an 
edge case worth investigating separately.

This means we are not dealing with random missingness.
People who are unemployed, retired, or outside the formal economy may systematically lack both fields. 
Imputing these blindly would introduce bias.

**Strategy:** create a `missing` category for both columns rather than imputing because the absence of information is itself informative.

In [32]:
# rows where occupation is missing but workclass is not
edge_cases = train[(train["occupation"] == "?") & (train["workclass"] != "?")]
print(edge_cases["workclass"].value_counts())

workclass
Never-worked    7
Name: count, dtype: int64


In [35]:
print(train.workclass.value_counts())

workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64


In [36]:
print(train.occupation.value_counts())

occupation
Prof-specialty       4140
Craft-repair         4099
Exec-managerial      4066
Adm-clerical         3770
Sales                3650
Other-service        3295
Machine-op-inspct    2002
?                    1843
Transport-moving     1597
Handlers-cleaners    1370
Farming-fishing       994
Tech-support          928
Protective-serv       649
Priv-house-serv       149
Armed-Forces            9
Name: count, dtype: int64


### Observation 4 — Never-worked Edge Case

The 7 rows where `occupation` is missing and `workclass` is all labeled `Never-worked`. This makes complete sense — someone who has never worked has no occupation to report. This is not a data quality issue, it is a logically consistent pattern.

This confirms our strategy: treat `?` as a meaningful category called `Unknown` rather than imputing. For `Never-worked` rows, occupation 
will be set to `Never-worked` to maintain consistency.

### Check test set

In [37]:
print("Missing values in test set:")
missing_test = (test == "?").sum()
print(missing_test[missing_test > 0])
print()
print("As percentage of total rows:")
print((missing_test[missing_test > 0] / len(test) * 100).round(2))

Missing values in test set:
workclass         963
occupation        966
native-country    274
dtype: int64

As percentage of total rows:
workclass         5.91
occupation        5.93
native-country    1.68
dtype: float64


### Observation 5 — Missing Pattern Consistent Across Train and Test

Test set shows the same structural missing pattern:
- **workclass:** 5.91% missing (train: 5.64%)
- **occupation:** 5.93% missing (train: 5.66%)
- **native-country:** 1.68% missing (train: 1.79%)

Proportions are stable across splits — this is not a sampling artifact.
The missingness is a consistent property of the data collection process, further supporting the decision to treat `?` as a meaningful category rather than imputing.

## 4. Leakage and Proxy Risk

Some features may act as proxies for protected attributes, not leakage in the traditional sense, but a fairness risk. 
We identify these before modeling.

In [38]:
# occupation distribution by sex -- proxy risk check
print(train.groupby("sex")["occupation"].value_counts(normalize=True).round(3))

sex     occupation       
Female  Adm-clerical         0.236
        Other-service        0.167
        Prof-specialty       0.141
        Sales                0.117
        Exec-managerial      0.108
        ?                    0.078
        Machine-op-inspct    0.051
        Tech-support         0.032
        Craft-repair         0.021
        Handlers-cleaners    0.015
        Priv-house-serv      0.013
        Transport-moving     0.008
        Protective-serv      0.007
        Farming-fishing      0.006
Male    Craft-repair         0.178
        Exec-managerial      0.133
        Prof-specialty       0.120
        Sales                0.110
        Transport-moving     0.069
        Other-service        0.069
        Machine-op-inspct    0.067
        Adm-clerical         0.057
        Handlers-cleaners    0.055
        ?                    0.046
        Farming-fishing      0.043
        Tech-support         0.027
        Protective-serv      0.026
        Armed-Forces         